In [1]:
K = 10
TEST_PERCENTAGE = 0.20
LEARNING_RATE = 0.25
LOSS_FUNCTION = 'bpr'
NO_COMPONENTS = 20
NO_EPOCHS = 20
NO_THREADS = 16
ITEM_ALPHA = 1e-6
USER_ALPHA = 1e-6
SEED = 42

In [2]:
import time
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

import lightfm
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, *args):
        self.interval = time.perf_counter() - self.start

print(f"NumPy: {np.__version__} | LightFM: {lightfm.__version__}")

NumPy: 2.4.4 | LightFM: 1.17


In [3]:
def _get_ranks(model, test_interactions, train_interactions, item_features=None, num_threads=2):
    return model.predict_rank(
        test_interactions=test_interactions,
        train_interactions=train_interactions,
        item_features=item_features,
        num_threads=num_threads
    )

def eval_ndcg_map(ranks, test_interactions, k=10):
    """Calcula nDCG@K y MAP@K a partir de una matriz de rankings sparse."""
    ranks_csr = ranks.tocsr()
    test_csr = test_interactions.tocsr()
    ndcgs, aps = [], []
    for uid in range(test_interactions.shape[0]):
        test_items = test_csr[uid].indices
        if len(test_items) == 0:
            continue
        user_ranks = ranks_csr[uid]
        dcg = 0.0
        hits_at_rank = []
        for item_idx in test_items:
            rank = user_ranks[0, item_idx]
            if 0 < rank <= k:
                dcg += 1.0 / np.log2(rank + 1)
                hits_at_rank.append(int(rank))
        ideal_k = min(k, len(test_items))
        idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_k))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
        hits_at_rank.sort()
        ap = sum((i + 1) / r for i, r in enumerate(hits_at_rank))
        n_rel = min(len(test_items), k)
        aps.append(ap / n_rel if n_rel > 0 else 0.0)
    return np.mean(ndcgs) if ndcgs else 0.0, np.mean(aps) if aps else 0.0

def eval_sellos(ranks, test_interactions, recipe_sellos_dict, iid_map, k=10):
    """Calcula S@K y SS@K a partir de una matriz de rankings sparse."""
    inv_iid = {v: kk for kk, v in iid_map.items()}
    ranks_csr = ranks.tocsr()
    test_csr = test_interactions.tocsr()
    s_list, sf_list = [], []
    for uid in range(test_interactions.shape[0]):
        if test_csr[uid].nnz == 0:
            continue
        user_ranks = np.asarray(ranks_csr[uid].todense()).flatten()
        top_k_idx = np.where((user_ranks > 0) & (user_ranks <= k))[0]
        if len(top_k_idx) == 0:
            continue
        sellos = [recipe_sellos_dict.get(inv_iid.get(idx, -1), 0) for idx in top_k_idx]
        s_list.append(np.mean(sellos))
        sf_list.append(np.mean([1 if s == 0 else 0 for s in sellos]))
    return (
        np.mean(s_list) if s_list else 0.0,
        np.mean(sf_list) if sf_list else 0.0
    )

# Carga de datos

In [4]:
import kagglehub
local_dataset_path = kagglehub.dataset_download('irkaal/foodcom-recipes-and-reviews')
reviews = pd.read_csv(os.path.join(local_dataset_path, 'reviews.csv'))

recipes = pd.read_csv(os.path.join('..', 'Dataset_recetas', 'recipes_final_consolidado', 'recipes_final_consolidado.csv'))
recipes['ExtractedServingSize'] = recipes['ExtractedServingSize'].str.extract(r'\((.*?)\)').astype(float)

print(f"Reviews: {len(reviews):,} | Recipes: {len(recipes):,}")

Reviews: 1,401,982 | Recipes: 522,359


In [5]:
review_counts = reviews['AuthorId'].value_counts()
little_review_users = review_counts[review_counts == 1]

reviews_filtrado = reviews[~reviews['AuthorId'].isin(little_review_users.index)]
recipes_filtrado = recipes[recipes['RecipeId'].isin(reviews_filtrado['RecipeId'].unique())]
recipes_filtrado = recipes_filtrado[recipes_filtrado['ExtractedServingSize'] > 0].copy()
reviews_filtrado = reviews_filtrado[reviews_filtrado['RecipeId'].isin(recipes_filtrado['RecipeId'].unique())].copy()


print(f"Reviews filtrado: {len(reviews_filtrado):,} | Recipes filtrado: {len(recipes_filtrado):,}")

Reviews filtrado: 1,199,589 | Recipes filtrado: 257,707


# Train / Test split

In [6]:
np.random.seed(SEED)

dataset_base = Dataset()
dataset_base.fit(
    users=reviews_filtrado['AuthorId'].unique(),
    items=reviews_filtrado['RecipeId'].unique()
)

interactions, weights = dataset_base.build_interactions(
    reviews_filtrado[['AuthorId', 'RecipeId', 'Rating']].itertuples(index=False, name=None)
)

uids, iids, vals = sp.find(interactions)
shuffle_idx = np.random.permutation(len(uids))
uids, iids, vals = uids[shuffle_idx], iids[shuffle_idx], vals[shuffle_idx]

cutoff = int((1 - TEST_PERCENTAGE) * len(uids))
n_users, n_items = interactions.shape

train_interactions = sp.coo_matrix(
    (vals[:cutoff], (uids[:cutoff], iids[:cutoff])),
    shape=(n_users, n_items)
).tocsr()

test_interactions = sp.coo_matrix(
    (vals[cutoff:], (uids[cutoff:], iids[cutoff:])),
    shape=(n_users, n_items)
).tocsr()

uid_map, _, iid_map, _ = dataset_base.mapping()

print(f"Train: {train_interactions.nnz:,} | Test: {test_interactions.nnz:,}")
print(f"Users: {n_users:,} | Items: {n_items:,}")

Train: 959,671 | Test: 239,918
Users: 72,098 | Items: 257,707


# Modelo sin sellos (base)

In [7]:
model_base = LightFM(
    loss=LOSS_FUNCTION,
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=np.random.RandomState(SEED)
)

with Timer() as t:
    model_base.fit(
        interactions=train_interactions,
        epochs=NO_EPOCHS,
        num_threads=NO_THREADS,
        verbose=True
    )
print(f'Entrenamiento: {t.interval:.1f}s')

Epoch: 100%|██████████| 20/20 [00:08<00:00,  2.24it/s]

Entrenamiento: 9.2s


In [8]:
with Timer() as t:
    eval_precision_sin = precision_at_k(
        model_base, test_interactions, train_interactions, k=K, num_threads=NO_THREADS
    ).mean()
    eval_recall_sin = recall_at_k(
        model_base, test_interactions, train_interactions, k=K, num_threads=NO_THREADS
    ).mean()
    eval_auc_sin = auc_score(
        model_base, test_interactions, train_interactions, num_threads=NO_THREADS
    ).mean()

    ranks_sin = _get_ranks(model_base, test_interactions, train_interactions, num_threads=NO_THREADS)
    eval_ndcg_sin, eval_map_sin = eval_ndcg_map(ranks_sin, test_interactions, k=K)

if (eval_precision_sin + eval_recall_sin) > 0:
    eval_f1_sin = 2 * (eval_precision_sin * eval_recall_sin) / (eval_precision_sin + eval_recall_sin)
else:
    eval_f1_sin = 0.0

print(f"Evaluación: {t.interval:.1f}s")
print(f"\nPrecision@K:  {eval_precision_sin:.6f}")
print(f"Recall@K:     {eval_recall_sin:.6f}")
print(f"F1@K:         {eval_f1_sin:.6f}")
print(f"MAP@K:        {eval_map_sin:.6f}")
print(f"NDCG@K:       {eval_ndcg_sin:.6f}")
print(f"AUC ROC:      {eval_auc_sin:.6f}")

Evaluación: 316.4s

Precision@K:  0.001872
Recall@K:     0.005779
F1@K:         0.002827
MAP@K:        0.001862
NDCG@K:       0.003292
AUC ROC:      0.680566


# Sellos (Ley 20.606 Fase 3)

In [9]:
high_sugar = 10
high_saturated_fat = 4
high_calories = 275
high_sodium = 400

recipes_filtrado['IsHighSugar'] = ((recipes_filtrado['SugarContent'] / recipes_filtrado['ExtractedServingSize']) * 100) >= high_sugar
recipes_filtrado['IsHighSaturatedFat'] = ((recipes_filtrado['SaturatedFatContent'] / recipes_filtrado['ExtractedServingSize']) * 100) >= high_saturated_fat
recipes_filtrado['IsHighCalories'] = ((recipes_filtrado['Calories'] / recipes_filtrado['ExtractedServingSize']) * 100) >= high_calories
recipes_filtrado['IsHighSodium'] = ((recipes_filtrado['SodiumContent'] / recipes_filtrado['ExtractedServingSize']) * 100) >= high_sodium

recipes_filtrado['TotalSellos'] = (
    recipes_filtrado['IsHighSugar'].astype(int) +
    recipes_filtrado['IsHighSaturatedFat'].astype(int) +
    recipes_filtrado['IsHighCalories'].astype(int) +
    recipes_filtrado['IsHighSodium'].astype(int)
)
recipe_sellos_dict = dict(zip(recipes_filtrado['RecipeId'], recipes_filtrado['TotalSellos']))

print(f"Distribución de sellos:")
print(recipes_filtrado['TotalSellos'].value_counts().sort_index())

Distribución de sellos:
TotalSellos
0    117799
1     64099
2     35997
3     34990
4      4822
Name: count, dtype: int64


In [10]:
with Timer() as t:
    eval_s_k_sin, eval_ss_k_sin = eval_sellos(
        ranks_sin, test_interactions, recipe_sellos_dict, iid_map, k=K
    )

print(f"S@K y SS@K calculados en {t.interval:.1f}s")
print(f"S@K  (menor=mejor):  {eval_s_k_sin:.6f}")
print(f"SS@K (mayor=mejor):  {eval_ss_k_sin:.6f}")

S@K y SS@K calculados en 16.0s
S@K  (menor=mejor):  1.099630
SS@K (mayor=mejor):  0.404149


# Modelo con sellos (híbrido)

In [11]:
sello_features = ['IsHighSugar', 'IsHighSaturatedFat', 'IsHighCalories', 'IsHighSodium']

dataset_con_sellos = Dataset()
dataset_con_sellos.fit(
    users=reviews_filtrado['AuthorId'].unique(),
    items=reviews_filtrado['RecipeId'].unique(),
    item_features=sello_features
)

interactions2, weights2 = dataset_con_sellos.build_interactions(
    reviews_filtrado[['AuthorId', 'RecipeId', 'Rating']].itertuples(index=False, name=None)
)

recipes_in_reviews = recipes_filtrado[recipes_filtrado['RecipeId'].isin(reviews_filtrado['RecipeId'].unique())]
item_features_list = []
for _, row in recipes_in_reviews.iterrows():
    feats = [f for f in sello_features if row.get(f, False)]
    if feats:
        item_features_list.append((row['RecipeId'], feats))

item_features = dataset_con_sellos.build_item_features(item_features_list)
print(f"item_features shape: {item_features.shape}")

uid_map2, _, iid_map2, _ = dataset_con_sellos.mapping()

uids2, iids2, vals2 = sp.find(interactions2)
shuffle_idx2 = np.random.permutation(len(uids2))
uids2, iids2, vals2 = uids2[shuffle_idx2], iids2[shuffle_idx2], vals2[shuffle_idx2]

cutoff2 = int((1 - TEST_PERCENTAGE) * len(uids2))
n_users2, n_items2 = interactions2.shape

train_interactions2 = sp.coo_matrix(
    (vals2[:cutoff2], (uids2[:cutoff2], iids2[:cutoff2])),
    shape=(n_users2, n_items2)
).tocsr()

test_interactions2 = sp.coo_matrix(
    (vals2[cutoff2:], (uids2[cutoff2:], iids2[cutoff2:])),
    shape=(n_users2, n_items2)
).tocsr()

print(f"Train: {train_interactions2.nnz:,} | Test: {test_interactions2.nnz:,}")

item_features shape: (257707, 257711)
Train: 959,671 | Test: 239,918


In [12]:
model2 = LightFM(
    loss=LOSS_FUNCTION,
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=np.random.RandomState(SEED)
)

with Timer() as t:
    model2.fit(
        interactions=train_interactions2,
        item_features=item_features,
        epochs=NO_EPOCHS,
        num_threads=NO_THREADS,
        verbose=True
    )
print(f'Entrenamiento: {t.interval:.1f}s')

Epoch: 100%|██████████| 20/20 [00:25<00:00,  1.26s/it]

Entrenamiento: 25.5s


In [13]:
with Timer() as t:
    eval_precision_con = precision_at_k(
        model2, test_interactions2, train_interactions2, k=K,
        item_features=item_features, num_threads=NO_THREADS
    ).mean()
    eval_recall_con = recall_at_k(
        model2, test_interactions2, train_interactions2, k=K,
        item_features=item_features, num_threads=NO_THREADS
    ).mean()
    eval_auc_con = auc_score(
        model2, test_interactions2, train_interactions2,
        item_features=item_features, num_threads=NO_THREADS
    ).mean()

    ranks_con = _get_ranks(
        model2, test_interactions2, train_interactions2,
        item_features=item_features, num_threads=NO_THREADS
    )
    eval_ndcg_con, eval_map_con = eval_ndcg_map(ranks_con, test_interactions2, k=K)
    eval_s_k_con, eval_ss_k_con = eval_sellos(
        ranks_con, test_interactions2, recipe_sellos_dict, iid_map2, k=K
    )

if (eval_precision_con + eval_recall_con) > 0:
    eval_f1_con = 2 * (eval_precision_con * eval_recall_con) / (eval_precision_con + eval_recall_con)
else:
    eval_f1_con = 0.0

print(f"Evaluación: {t.interval:.1f}s")
print(f"\nPrecision@K:  {eval_precision_con:.6f}")
print(f"Recall@K:     {eval_recall_con:.6f}")
print(f"F1@K:         {eval_f1_con:.6f}")
print(f"MAP@K:        {eval_map_con:.6f}")
print(f"NDCG@K:       {eval_ndcg_con:.6f}")
print(f"AUC ROC:      {eval_auc_con:.6f}")
print(f"S@K:          {eval_s_k_con:.6f}")
print(f"SS@K:         {eval_ss_k_con:.6f}")

Evaluación: 375.6s

Precision@K:  0.001026
Recall@K:     0.002613
F1@K:         0.001474
MAP@K:        0.000825
NDCG@K:       0.001647
AUC ROC:      0.651258
S@K:          0.639286
SS@K:         0.575238


# Comparación final

In [14]:
resultados = pd.DataFrame({
    'Métrica': ['Precision@K', 'Recall@K', 'F1@K', 'MAP@K', 'NDCG@K', 'AUC ROC',
                'S@K (Promedio Sellos)', 'SS@K (% Sin Sellos)'],
    'Sin Sellos (Base)': [eval_precision_sin, eval_recall_sin, eval_f1_sin,
                          eval_map_sin, eval_ndcg_sin, eval_auc_sin,
                          eval_s_k_sin, eval_ss_k_sin],
    'Con Sellos (Híbrido)': [eval_precision_con, eval_recall_con, eval_f1_con,
                              eval_map_con, eval_ndcg_con, eval_auc_con,
                              eval_s_k_con, eval_ss_k_con]
})

print("=================== COMPARATIVA DE RENDIMIENTO Y SALUD ===================")
print(resultados.to_string(index=False, formatters={
    'Sin Sellos (Base)': '{:,.6f}'.format,
    'Con Sellos (Híbrido)': '{:,.6f}'.format
}))
print("==========================================================================")

=================== COMPARATIVA DE RENDIMIENTO Y SALUD ===================
              Métrica Sin Sellos (Base) Con Sellos (Híbrido)
          Precision@K          0.001872             0.001026
             Recall@K          0.005779             0.002613
                 F1@K          0.002827             0.001474
                MAP@K          0.001862             0.000825
               NDCG@K          0.003292             0.001647
              AUC ROC          0.680566             0.651258
S@K (Promedio Sellos)          1.099630             0.639286
  SS@K (% Sin Sellos)          0.404149             0.575238
